# Chapter 4 -- Context Engineering (Practice)

Work through this notebook **after reading** `notes/ch04-context-engineering.md`. This chapter treats the context window as a finite, actively degrading resource and builds the four moves from notes Section 4 (write, select, compress, isolate) into working code: a token-accounting `ContextManager`, three eviction strategies (FIFO, tool-result clearing, summarize-oldest), microcompaction, and instruction re-injection.

The task is a deterministic, fully offline simulation of a 20-step agent run -- no API key or model call needed for any exercise. A critical fact is planted early in the simulated trajectory; by the end you will measure, not just read about, which context-management choices actually keep it alive.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
import anthropic


def _find_and_load_env() -> None:
    """Walk up from the current working directory to find and load a repo-root .env file, if one exists."""
    here = Path.cwd()
    for parent in [here, *here.parents]:
        candidate = parent / ".env"
        if candidate.is_file():
            load_dotenv(candidate)
            return
    print("No .env file found -- copy .env.example to .env at the repo root to enable the real-model sections.")


_find_and_load_env()

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
MODEL_NAME = os.getenv("BEDROCK_MODEL_ID", "anthropic.claude-sonnet-5")


def test_connection(client, model_name):
    """Send a trivial ping to confirm the Bedrock connection actually works."""
    print(f"Testing connection to Bedrock (model={model_name})...")
    try:
        response = client.messages.create(
            model=model_name, max_tokens=10,
            messages=[{"role": "user", "content": "Reply with exactly the word: pong"}],
        )
        reply = next((b.text for b in response.content if b.type == "text"), "")
        status = "PASS" if "pong" in reply.lower() else f"unexpected reply: {reply!r}"
        print(f"  {status}")
    except Exception as exc:
        print(f"  Connection check FAILED: {type(exc).__name__}: {exc}")
        print()
        print("The rest of this notebook still works fully offline -- this cell")
        print("only matters for the optional real-model section at the end.")


if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
    print("AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY not set in .env -- skipping connection test.")
    print("The rest of this notebook still works fully offline.")
else:
    client = anthropic.AnthropicBedrockMantle(
        aws_access_key=AWS_ACCESS_KEY_ID, aws_secret_key=AWS_SECRET_ACCESS_KEY, aws_region=AWS_REGION,
    )
    test_connection(client, MODEL_NAME)


## The Shared Simulated Task

A 20-step agent run, simulated deterministically so every exercise below is checkable without an API key. Each step produces an assistant block (short) and a tool-result block (a simulated file read, ~900 characters). Step 3's tool result contains a **critical fact** -- a file path the rest of the (imagined) task depends on -- buried inside an otherwise ordinary-looking tool observation, exactly like notes Section 6's "the error you were mid-investigation" scenario, just with a fact instead of an error.

`estimate_tokens` reuses Chapter 3's `len(text) // 4` heuristic for consistency across this repository's notebooks.

In [ ]:
def estimate_tokens(text):
    """Rough len(text)//4 approximation -- NOT a real tokenizer (same heuristic as Chapter 3)."""
    return max(1, len(text) // 4)


def make_filler_text(n_chars, seed="Reading configuration file contents line by line for verification purposes. "):
    """Deterministic filler text of exactly n_chars, standing in for a real tool observation."""
    return (seed * (n_chars // len(seed) + 1))[:n_chars]


CRITICAL_FACT = (
    "CRITICAL: production config lives at 'configs/prod-v3.yaml' -- use "
    "this exact path for every remaining step."
)

N_STEPS = 20
TOOL_CHARS = 900
BUDGET = 3000  # trajectory-only token budget (fixed prefix tracked separately)

TRAJECTORY = []
for step in range(1, N_STEPS + 1):
    assistant_text = f"Step {step}: reasoning about next action, calling a tool."
    if step == 3:
        # The critical fact is padded to the SAME length as every other tool
        # block -- it must not survive later just because it happened to be
        # smaller than everything competing with it.
        pad = make_filler_text(TOOL_CHARS - len(CRITICAL_FACT) - 1)
        tool_text = CRITICAL_FACT + " " + pad
    else:
        tool_text = make_filler_text(TOOL_CHARS)
    TRAJECTORY.append({
        "step": step,
        "assistant": {"role": "assistant", "tag": f"assistant_{step}", "text": assistant_text, "tokens": estimate_tokens(assistant_text)},
        "tool": {"role": "tool", "tag": f"tool_{step}", "text": tool_text, "tokens": estimate_tokens(tool_text)},
    })

per_step_tokens = TRAJECTORY[0]["assistant"]["tokens"] + TRAJECTORY[0]["tool"]["tokens"]
print(f"{N_STEPS} steps built, ~{per_step_tokens} tokens/step, ~{per_step_tokens * N_STEPS} tokens if never evicted")
print(f"BUDGET = {BUDGET} tokens (trajectory only) -- naive crossing point: step {BUDGET / per_step_tokens:.1f}")
print(f"Critical fact planted in step 3's tool result ({TRAJECTORY[2]['tool']['tokens']} tokens, same size as any other step).")


## Token Accounting: the `ContextManager`

`ContextManager` is this chapter's answer to notes Section 3's "write the budget down": it holds the currently **active** blocks (what would actually be sent to a model), applies microcompaction and an eviction strategy every step, optionally re-injects the critical fact if it's gone missing, and can print a per-step budget line -- the code equivalent of the budget table notes Section 12 built by hand.

In [ ]:
def total_tokens(active):
    return sum(b["tokens"] for b in active)


class ContextManager:
    """
    Wraps the per-step context-management pipeline: microcompaction (if
    enabled) -> eviction strategy -> re-injection (if enabled). `active` is
    the list of blocks that would actually be sent to a model right now --
    this is the thing every exercise below measures.
    """

    def __init__(self, budget, strategy_fn, use_microcompaction=False, use_reinjection=False, critical_fact=None):
        self.budget = budget
        self.strategy_fn = strategy_fn
        self.use_microcompaction = use_microcompaction
        self.use_reinjection = use_reinjection
        self.critical_fact = critical_fact
        self.active = []

    def step(self, new_blocks):
        if self.use_microcompaction:
            self.active = microcompact(self.active)
        self.active = self.strategy_fn(self.active, new_blocks, self.budget)
        if self.use_reinjection and self.critical_fact:
            self.active = reinject_if_missing(self.active, self.critical_fact)

    def print_budget_line(self, step_num):
        tokens = total_tokens(self.active)
        status = "OVER BUDGET" if tokens > self.budget else "ok"
        print(f"  step {step_num:2d}: blocks={len(self.active):3d}  tokens={tokens:5d}/{self.budget}  [{status}]")

    def fact_survived(self):
        return any(self.critical_fact in b["text"] for b in self.active)


print("ContextManager defined.")


## Two Given Eviction Strategies

`fifo_evict` drops the single oldest block, repeatedly, until back under budget -- the simplest possible policy, and notes Section 6/13's cautionary example of what happens with no judgment about *what* gets dropped. `tool_result_clearing` implements notes Section 7 directly: once a tool block is at least `min_age` steps old, its content is replaced with a short placeholder (a prune, keeping the block's presence in the transcript, discarding only its content) rather than removing the block outright.

In [ ]:
def fifo_evict(active, new_blocks, budget):
    """Drop the oldest block, repeatedly, until under budget. No judgment about what's dropped."""
    active = active + new_blocks
    while total_tokens(active) > budget and len(active) > 1:
        active = active[1:]
    return active


def tool_result_clearing(active, new_blocks, budget, min_age=3):
    """
    Notes Section 7: clear (don't remove) the content of old tool blocks
    once over budget. A block survives structurally -- only its content
    is replaced with a short placeholder.
    """
    active = active + new_blocks
    while total_tokens(active) > budget:
        candidates = [
            i for i, b in enumerate(active)
            if b["role"] == "tool" and not b.get("cleared") and (len(active) - 1 - i) >= min_age
        ]
        if not candidates:
            break  # nothing left old enough to clear
        i = candidates[0]
        cleared_block = dict(active[i])
        cleared_block["text"] = "[cleared: tool result no longer shown]"
        cleared_block["tokens"] = estimate_tokens(cleared_block["text"])
        cleared_block["cleared"] = True
        active = active[:i] + [cleared_block] + active[i + 1:]
    return active


print("Given strategies defined:", ["fifo_evict", "tool_result_clearing"])


## Exercise 1 -- Summarize-Oldest

Implement `summarize_oldest`: once over budget, take the oldest `chunk_size` blocks and replace them with a **single** synthetic summary block, freeing up the budget those blocks were consuming. Keep it simple -- the summary text just needs to name what was summarized (e.g. by joining each removed block's `tag`); it does **not** need to preserve their actual content verbatim. That's deliberate: notes Section 6 named this exact failure mode ("a why-decision compresses more easily than the reasoning survives") and the comparison in this notebook's final section measures its real consequence directly, rather than just asserting it.

In [ ]:
def summarize_oldest(active, new_blocks, budget, chunk_size=4):
    """
    Once over budget, replace the oldest `chunk_size` blocks with ONE
    summary block. Repeat until under budget or too few blocks remain
    to summarize further.

    Returns: the new active list.
    """
    active = active + new_blocks
    while total_tokens(active) > budget and len(active) > chunk_size:
        # TODO: take active[:chunk_size], build a short summary_text that
        # names those blocks (e.g. join their "tag" fields), build a
        # summary block {"role": "summary", "tag": ..., "text": summary_text,
        # "tokens": estimate_tokens(summary_text)}, and replace the first
        # chunk_size blocks in `active` with just that one summary block.
        raise NotImplementedError("TODO: implement summarize_oldest")
    return active


In [ ]:
small_active = [
    {"role": "tool", "tag": f"tool_{i}", "text": make_filler_text(400), "tokens": estimate_tokens(make_filler_text(400))}
    for i in range(6)
]
before_len, before_tokens = len(small_active), total_tokens(small_active)
result = summarize_oldest(small_active, [], budget=300, chunk_size=4)

assert len(result) < before_len, "summarize_oldest should reduce the number of blocks when over budget"
assert total_tokens(result) < before_tokens, "summarize_oldest should reduce total tokens when it fires"
assert any(b["role"] == "summary" for b in result), "expected at least one block with role=='summary'"
summary_block = next(b for b in result if b["role"] == "summary")
assert "tool_0" in summary_block["text"] and "tool_3" in summary_block["text"], "summary should name the blocks it replaced"

# When already under budget, nothing should be touched.
untouched = summarize_oldest([], [{"role": "tool", "tag": "t", "text": "short", "tokens": 2}], budget=1000, chunk_size=4)
assert len(untouched) == 1 and untouched[0]["tag"] == "t", "should not summarize when already under budget"

print("Exercise 1 PASSED -- summarize_oldest replaces the oldest chunk with one summary block")
print("when over budget, and leaves things alone when already under budget.")


## Exercise 2 -- Microcompaction

Implement `microcompact`: notes Section 5 distinguished this from full compaction specifically because it's **cheap, continuous, and proactive** -- it runs on every step regardless of whether the total budget is currently exceeded, targeting individually oversized tool blocks before they ever contribute to a budget crisis. Replace any tool block above `threshold` tokens (that hasn't already been cleared or microcompacted) with a short placeholder noting it was "stored externally".

In [ ]:
def microcompact(active, threshold=200):
    """
    Proactively replace any oversized tool block with a short placeholder,
    independent of whether the total budget is currently exceeded.
    Skips blocks already cleared (tool_result_clearing) or microcompacted.

    Returns: the new active list (same length -- this never removes blocks,
    only shrinks their content).
    """
    out = []
    for b in active:
        # TODO: if b["role"]=="tool" and not already cleared/microcompacted
        # and b["tokens"] > threshold, append a copy of b with its "text"
        # replaced by a short placeholder (mention b["tag"] and its
        # original token count), "tokens" recomputed via estimate_tokens,
        # and "microcompacted": True set. Otherwise append b unchanged.
        raise NotImplementedError("TODO: implement microcompact")
    return out


In [ ]:
big_block = {"role": "tool", "tag": "big", "text": make_filler_text(900), "tokens": estimate_tokens(make_filler_text(900))}
small_block = {"role": "tool", "tag": "small", "text": "ok", "tokens": estimate_tokens("ok")}
already_cleared = {"role": "tool", "tag": "cleared_one", "text": "[cleared: tool result no longer shown]", "tokens": 8, "cleared": True}

result = microcompact([big_block, small_block, already_cleared], threshold=200)

assert len(result) == 3, "microcompact should never remove a block, only shrink its content"
big_after = next(b for b in result if b["tag"] == "big")
assert big_after["tokens"] < big_block["tokens"], "the oversized block should have shrunk"
assert big_after.get("microcompacted") is True, "the oversized block should be flagged microcompacted"
assert "big" in big_after["text"], "the placeholder should reference the original block's tag"

small_after = next(b for b in result if b["tag"] == "small")
assert small_after == small_block, "a block already under threshold should be left completely unchanged"

cleared_after = next(b for b in result if b["tag"] == "cleared_one")
assert cleared_after == already_cleared, "an already-cleared block should be skipped, not double-processed"

print("Exercise 2 PASSED -- oversized tool blocks are shrunk to placeholders,")
print("small and already-cleared blocks are left untouched.")


## Exercise 3 -- Instruction Re-Injection

Implement `reinject_if_missing`: notes Section 6's concrete mitigation for compaction (or any eviction) silently dropping something durable. Check whether `critical_text` is still present anywhere in `active`; if it's gone, append a small reminder block containing it verbatim at the **end** (recency-favored, per notes Section 7). If it's still present, do nothing -- this must be idempotent, or repeated calls would pile up duplicate reminders forever.

In [ ]:
def reinject_if_missing(active, critical_text):
    """
    If `critical_text` is not present anywhere in `active`'s block text,
    append one reminder block containing it verbatim. If it's already
    present, return `active` unchanged (idempotent -- never duplicate).

    Returns: the new active list.
    """
    # TODO: search active for any block whose "text" contains
    # critical_text. If found, return active unchanged. If not found,
    # return active + [a new block: {"role": "system_reminder",
    # "tag": "reinjected_critical_fact", "text": critical_text,
    # "tokens": estimate_tokens(critical_text)}].
    raise NotImplementedError("TODO: implement reinject_if_missing")


In [ ]:
missing_case = [{"role": "tool", "tag": "t1", "text": "unrelated content", "tokens": 3}]
result = reinject_if_missing(missing_case, CRITICAL_FACT)
assert len(result) == 2, "should append exactly one reminder block when the fact is missing"
assert result[-1]["text"] == CRITICAL_FACT, "the reminder should contain the fact verbatim"
assert result[-1]["role"] == "system_reminder"

present_case = [{"role": "tool", "tag": "t1", "text": f"some text including: {CRITICAL_FACT}", "tokens": 20}]
result2 = reinject_if_missing(present_case, CRITICAL_FACT)
assert len(result2) == 1, "should NOT append a reminder when the fact is already present"

# idempotency: calling it twice in a row must not duplicate the reminder
twice = reinject_if_missing(reinject_if_missing(missing_case, CRITICAL_FACT), CRITICAL_FACT)
assert len(twice) == 2, "calling reinject_if_missing twice should not add a second reminder"

print("Exercise 3 PASSED -- reinject_if_missing restores a missing critical fact,")
print("leaves it alone when already present, and is idempotent under repeated calls.")


## Comparing Strategies Across the Full 20-Step Run

First, one detailed trace: FIFO with no safety net, printed every step, so you can see exactly where the 3,000-token budget gets crossed (notes Section 12 solved this same crossing-point arithmetic by hand -- watch it happen here in code). Then, the real comparison: four configurations run start-to-finish, checked at the end for one thing only -- **did `CRITICAL_FACT` survive?**

In [ ]:
print("-" * 60)
print("DETAILED TRACE: FIFO, no re-injection")
print("-" * 60)

trace_manager = ContextManager(budget=BUDGET, strategy_fn=fifo_evict, critical_fact=CRITICAL_FACT)
for step_data in TRAJECTORY:
    trace_manager.step([step_data["assistant"], step_data["tool"]])
    trace_manager.print_budget_line(step_data["step"])

print()
print(f"Critical fact survived to the end? {trace_manager.fact_survived()}")


In [ ]:
def run_strategy(name, strategy_fn, use_microcompaction=False, use_reinjection=False):
    manager = ContextManager(
        budget=BUDGET, strategy_fn=strategy_fn,
        use_microcompaction=use_microcompaction, use_reinjection=use_reinjection,
        critical_fact=CRITICAL_FACT,
    )
    for step_data in TRAJECTORY:
        manager.step([step_data["assistant"], step_data["tool"]])
    survived = manager.fact_survived()
    print(f"{name:50s} survived={str(survived):5s}  final_blocks={len(manager.active):3d}  final_tokens={total_tokens(manager.active)}")
    return survived


print("-" * 60)
print("COMPARISON: does the critical fact from step 3 survive 20 steps?")
print("-" * 60)

results = {}
results["A) FIFO, no re-injection"] = run_strategy("A) FIFO, no re-injection", fifo_evict)
results["B) Tool-result clearing, no re-injection"] = run_strategy("B) Tool-result clearing, no re-injection", tool_result_clearing)
results["C) Summarize-oldest, no re-injection"] = run_strategy("C) Summarize-oldest, no re-injection", summarize_oldest)
results["D) Tool-clearing + microcompaction + re-injection"] = run_strategy(
    "D) Tool-clearing + microcompaction + re-injection", tool_result_clearing,
    use_microcompaction=True, use_reinjection=True,
)

print()
baseline_names = list(results.keys())[:3]
n_failed = sum(1 for name in baseline_names if not results[name])
print(f"{n_failed} out of {len(baseline_names)} eviction-only strategies (A, B, C) lost the critical fact.")
print("Adding re-injection (D) recovers it regardless of which eviction strategy underlies it --")
print("this is notes Section 6's claim, measured rather than just asserted.")


## Optional -- Try the Real `context_management` Beta

Notes Section 5 and Section 7 described two real, currently-documented Anthropic betas: `compact_20260112` (full compaction) and `clear_tool_uses_20250919` (tool-result clearing). This section attempts a real call using `clear_tool_uses_20250919` via `AnthropicBedrockMantle` (Claude Sonnet, through AWS Bedrock). Beta feature availability on Bedrock can lag the direct Anthropic API -- a clean failure here (unsupported beta, auth) is an expected, informative outcome, not a bug in this notebook.

In [ ]:
RUN_REAL_CONTEXT_DEMO = False


def run_real_context_demo():
    if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
        print("Skipping real context-management demo: AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY not set in .env.")
        return

    real_client = anthropic.AnthropicBedrockMantle(
        aws_access_key=AWS_ACCESS_KEY_ID, aws_secret_key=AWS_SECRET_ACCESS_KEY, aws_region=AWS_REGION,
    )
    try:
        response = real_client.beta.messages.create(
            model=MODEL_NAME,
            max_tokens=200,
            betas=["context-management-2025-06-27"],
            context_management={"edits": [{"type": "clear_tool_uses_20250919"}]},
            messages=[{"role": "user", "content": "Say 'context management beta reachable' and nothing else."}],
        )
    except Exception as exc:
        print(f"Real context-management call failed: {type(exc).__name__}: {exc}")
        print("(Expected if this beta isn't yet available on this account/platform.)")
        return

    text = next((b.text for b in response.content if b.type == "text"), "")
    print(f"Real response: {text}")
    print(f"stop_reason={response.stop_reason}")


if RUN_REAL_CONTEXT_DEMO:
    run_real_context_demo()
else:
    print("RUN_REAL_CONTEXT_DEMO is False -- running in offline/simulated mode only.")
    print("Flip it to True to attempt the real context_management beta via Bedrock.")


## Key Takeaways

You've built and measured, not just read about, notes Section 4's four operations: `microcompact` and `tool_result_clearing` are compress; `summarize_oldest` is also compress, in its heaviest form; `reinject_if_missing` implements Section 6's mitigation directly. The comparison in this notebook's core result is worth remembering on its own: **all three eviction strategies, used alone, silently lost a fact planted only 3 steps into a 20-step run** -- not because any of them was badly written, but because none of them had any concept of "this specific thing matters more than its age or size." Re-injection isn't a smarter eviction strategy; it's an explicit, cheap safety net layered on top of whatever eviction strategy you choose, and it's the one thing in this notebook that reliably worked.

**Connection forward:** Chapter 5 steps back from context specifically to the full harness surrounding Chapter 2's loop -- of which everything built in Chapters 3 and 4 (tool design, the `ContextManager` here) is one pluggable component among roughly ten, each independently implementable, testable, and versionable.